In [1]:
# Run this code every time when you're actively developing modules in .py files.  It's not needed if you aren't making modules
#
## this code is necessary for making sure that any modules we load are updated here 
## when their source code .py files are modifiedz

%load_ext autoreload
%autoreload 2

In [2]:
# Setup code -- Run only once after cloning!!! 
#
# this code downloads the data from its source to the `data/00-raw/` directory
# if the data hasn't updated you don't need to do this again!

# if you don't already have these packages (you should!) uncomment this line
# %pip install requests tqdm
import pandas as pd
import sys
sys.path.append('./modules') # this tells python where to look for modules to import

import get_data # this is where we get the function we need to download data

# replace the urls and filenames in this list with your actual datafiles
# yes you can use Google drive share links or whatever
# format is a list of dictionaries; 
# each dict has keys of 
#   'url' where the resource is located
#   'filename' for the local filename where it will be stored 
datafiles = [
    {'url': 'https://www.sandiego.gov/sites/default/files/2024-04/nibrs-crime-rates-per-1000-residents-by-neighborhood-2023.pdf', 'filename':'nibrs-crime-rates-per-1000-residents-by-neighborhood-2023.pdf'},
    {'url': 'https://www.sandiego.gov/sites/default/files/2025-03/nibrs-crime-rates-per-1000-residents-2024.pdf', 'filename':'nibrs-crime-rates-per-1000-residents-2024.pdf'},
    {'url': 'https://www.sandiego.gov/sites/default/files/2026-02/nibrs-crime-rates-per-1000-residents-2025.pdf', 'filename':'nibrs-crime-rates-per-1000-residents-2025.pdf'}
]

get_data.get_raw(datafiles,destination_directory='data/00-raw/')

Overall Download Progress:  33%|███▎      | 1/3 [00:00<00:00,  2.43it/s]                                             

Successfully downloaded: nibrs-crime-rates-per-1000-residents-by-neighborhood-2023.pdf



Overall Download Progress:  67%|██████▋   | 2/3 [00:00<00:00,  3.21it/s]                             

Successfully downloaded: nibrs-crime-rates-per-1000-residents-2024.pdf



Overall Download Progress: 100%|██████████| 3/3 [00:00<00:00,  3.15it/s]                             

Successfully downloaded: nibrs-crime-rates-per-1000-residents-2025.pdf


In [3]:
%pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pdfplumber
import pandas as pd
import os
import re

def extract_category(pdf_path, year, category_name='Persons'):
    all_data = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            if year == '2023' or year == '2024':
                table = page.extract_table(table_settings={
                    "vertical_strategy": 'text', 
                    "horizontal_strategy": 'text'
                })
            else:
                table = page.extract_table()
            if table:
                if category_name == 'Persons' and 8 <= len(table[0]) <= 9:
                    all_data.extend(table)
                    column_names = ['Neighborhood', 'Homicide', 'Kidnapping/Abduction', 'Sexual Assault', 'Aggravated Assault', 'Simple Assault', 'Intimidation', 'Other Sex Offenses']
                elif category_name == 'Property' and len(table[0]) >= 10:
                    all_data.extend(table)
                    column_names = ['Neighborhood', 'Robbery', 'Arson', 'Burglary/Breaking & Entering', 'Larceny', 'Theft From Motor Vehicle', 'Motor Vehicle Theft', 'Fraud/Counterfeiting', 'Stolen Property Offenses', 'Vandalism']
                elif category_name == 'Society' and len(table[0]) <= 7:
                    all_data.extend(table)
                    column_names = ['Neighborhood', 'Drug/Narcotic Offenses', 'Pornography/Obscene Material', 'Gambling Offenses', 'Prostitution', 'Weapon Law Violations', 'Animal Cruelty']

    df = pd.DataFrame(all_data[1:], columns=column_names)
    if year == '2023' and category_name == 'Society':
        df = pd.DataFrame(all_data[4:], columns=column_names)

    df = df[~(df['Neighborhood'] == '') & (df['Neighborhood'] != 'Neighborhood')].reset_index().drop(columns=['index'])
    
    def clean_neighborhood(x):
        x = str(x).strip()
        x = re.sub(r"\s+", " ", x)
        return x.upper()

    df["Neighborhood"] = df["Neighborhood"].apply(clean_neighborhood)

    for col in df.columns[1:]:
        df[col] = df[col].str.strip().astype(float)

    return df

In [5]:
def find_empty_columns(df, df_name):
    empty_columns = []
    # print(f"\n--- Null % report for {df_name} ---") 
    for col in df.columns:
        pct_null = ((df.shape[0] - df[col].count()) / df.shape[0]) * 100
        if pct_null > 5:
            empty_columns.append(col)
        # print(f'{col}: {round(pct_null, 2)}% null\n----------------------')
    return empty_columns

In [6]:
def clean_crime_data(pdf_path, year):
    assert os.path.exists(pdf_path), f"Missing file: {pdf_path}"

    print(f"Cleaning {year} crime rates data\n")
    
    persons_df = extract_category(pdf_path, year, 'Persons')
    property_df = extract_category(pdf_path, year, 'Property')
    society_df = extract_category(pdf_path, year, 'Society')

    print("Shape of persons_df:", persons_df.shape)
    print("Shape of property_df:", property_df.shape)
    print("Shape of society_df:", society_df.shape)
    print('\n')

    clean_persons_df = persons_df.dropna(how='all', axis=0).dropna(how='all', axis=1)
    clean_property_df = property_df.dropna(how='all', axis=0).dropna(how='all', axis=1)
    clean_society_df  = society_df.dropna(how='all', axis=0).dropna(how='all', axis=1)

    empty_cols_persons = find_empty_columns(clean_persons_df,  "clean_persons_df")
    empty_cols_property = find_empty_columns(clean_property_df, "clean_property_df")
    empty_cols_society = find_empty_columns(clean_society_df,  "clean_society_df")

    # print(f"\nDropping empty columns (if any)\n")

    clean_persons_df = clean_persons_df.drop(columns=empty_cols_persons, errors='ignore')
    clean_property_df = clean_property_df.drop(columns=empty_cols_property, errors='ignore')
    clean_society_df = clean_society_df.drop(columns=empty_cols_society, errors='ignore')

    crime_df = (clean_persons_df
            .merge(clean_property_df, on="Neighborhood", how="inner")
            .merge(clean_society_df, on="Neighborhood", how="inner"))

    print("Merged crime_df shape:", crime_df.shape)
    print("Neighborhoods persons:", clean_persons_df["Neighborhood"].nunique())
    print("Neighborhoods property:", clean_property_df["Neighborhood"].nunique())
    print("Neighborhoods society:", clean_society_df["Neighborhood"].nunique())
    print("Neighborhoods merged:", crime_df["Neighborhood"].nunique())
    print('\n')
    
    # missing_from_merge = (
    #     set(clean_persons_df["Neighborhood"])
    #     - set(crime_df["Neighborhood"])
    # )
    # print("Neighborhoods lost in merge\t", len(missing_from_merge))

    print("Total number of nulls:", crime_df.isna().sum().sum())

    return crime_df

In [7]:
def upload_cleaned_crime_df(df, year): 
    folder_name = os.path.join('data', '02-processed')
    file_name = f'clean_nibrs_crime_rates_{year}.csv'
    full_path = os.path.join(folder_name, file_name)

    os.makedirs(folder_name, exist_ok=True)
    df.to_csv(full_path, index=False)
    print(f'Data successfully saved to "{full_path}"')

In [8]:
crime_df_2025 = clean_crime_data('data/00-raw/nibrs-crime-rates-per-1000-residents-2025.pdf', '2025')

Cleaning 2025 crime rates data

Shape of persons_df: (125, 8)
Shape of property_df: (125, 10)
Shape of society_df: (125, 7)


Merged crime_df shape: (125, 23)
Neighborhoods persons: 125
Neighborhoods property: 125
Neighborhoods society: 125
Neighborhoods merged: 125


Total number of nulls: 0


In [9]:
crime_df_2025.head()
# upload_cleaned_crime_df(crime_df_2025, '2025')

,Neighborhood,Homicide,Kidnapping/Abduction,Sexual Assault,Aggravated Assault,Simple Assault,Intimidation,Other Sex Offenses,Robbery,Arson,...,Motor Vehicle Theft,Fraud/Counterfeiting,Stolen Property Offenses,Vandalism,Drug/Narcotic Offenses,Pornography/Obscene Material,Gambling Offenses,Prostitution,Weapon Law Violations,Animal Cruelty
0,ADAMS NORTH,0.00,0.33,0.50,0.66,1.49,0.00,0.00,0.33,0.17,...,1.00,1.99,0.17,2.16,0.83,0.0,0.0,0.0,0.50,0.0
1,ALLIED GARDENS,0.00,0.00,0.16,0.73,2.04,0.33,0.00,0.00,0.08,...,1.38,2.77,0.08,2.44,0.65,0.0,0.0,0.0,0.24,0.0
2,ALTA VISTA,0.00,0.00,0.00,2.59,2.59,0.00,0.00,0.00,0.00,...,0.37,0.74,0.00,0.74,0.00,0.0,0.0,0.0,0.37,0.0
3,AZALEA/HOLLYWOOD PARK,0.00,0.00,1.22,4.56,4.87,0.91,0.00,0.00,0.00,...,3.35,1.83,0.91,4.56,6.39,0.0,0.3,0.0,0.91,0.0
4,BALBOA PARK,2.13,4.26,21.32,100.21,91.68,21.32,2.13,12.79,4.26,...,8.53,8.53,2.13,74.63,151.39,0.0,0.0,0.0,25.59,0.0


In [10]:
crime_df_2024 = clean_crime_data('data/00-raw/nibrs-crime-rates-per-1000-residents-2024.pdf', '2024')

Cleaning 2024 crime rates data

Shape of persons_df: (125, 8)
Shape of property_df: (125, 10)
Shape of society_df: (125, 7)


Merged crime_df shape: (125, 23)
Neighborhoods persons: 125
Neighborhoods property: 125
Neighborhoods society: 125
Neighborhoods merged: 125


Total number of nulls: 0


In [11]:
crime_df_2024.head()
# upload_cleaned_crime_df(crime_df_2024, '2024')

,Neighborhood,Homicide,Kidnapping/Abduction,Sexual Assault,Aggravated Assault,Simple Assault,Intimidation,Other Sex Offenses,Robbery,Arson,...,Motor Vehicle Theft,Fraud/Counterfeiting,Stolen Property Offenses,Vandalism,Drug/Narcotic Offenses,Pornography/Obscene Material,Gambling Offenses,Prostitution,Weapon Law Violations,Animal Cruelty
0,ADAMS NORTH,0.0,0.2,0.2,1.2,2.3,0.0,0.0,0.7,0.0,...,2.7,1.3,0.5,3.7,0.3,0.0,0.0,0.0,0.5,0.0
1,ALLIED GARDENS,0.0,0.1,0.2,1.2,2.3,0.4,0.0,0.3,0.0,...,2.3,1.5,0.2,2.5,1.4,0.0,0.0,0.0,0.4,0.0
2,ALTA VISTA,0.0,0.4,0.0,2.2,1.9,0.0,0.0,0.0,0.0,...,1.1,0.4,0.0,1.9,0.4,0.0,0.0,0.0,0.0,0.0
3,AZALEA/HOLLYWOOD PARK,0.0,0.3,0.3,2.7,7.0,0.9,0.0,0.3,0.6,...,3.4,1.8,0.9,7.3,2.4,0.0,0.0,0.0,0.3,0.0
4,BALBOA PARK,2.1,0.0,10.7,100.2,78.9,17.1,0.0,14.9,10.7,...,21.3,17.1,10.7,81.0,206.8,0.0,0.0,0.0,10.7,0.0


In [12]:
crime_df_2023 = clean_crime_data('data/00-raw/nibrs-crime-rates-per-1000-residents-by-neighborhood-2023.pdf', '2023')

Cleaning 2023 crime rates data

Shape of persons_df: (126, 8)
Shape of property_df: (126, 10)
Shape of society_df: (126, 7)


Merged crime_df shape: (126, 23)
Neighborhoods persons: 126
Neighborhoods property: 126
Neighborhoods society: 126
Neighborhoods merged: 126


Total number of nulls: 0


In [13]:
crime_df_2023

,Neighborhood,Homicide,Kidnapping/Abduction,Sexual Assault,Aggravated Assault,Simple Assault,Intimidation,Other Sex Offenses,Robbery,Arson,...,Motor Vehicle Theft,Fraud/Counterfeiting,Stolen Property Offenses,Vandalism,Drug/Narcotic Offenses,Pornography/Obscene Material,Gambling Offenses,Prostitution,Weapon Law Violations,Animal Cruelty
0,ADAMS NORTH,0.0,0.0,0.0,1.5,3.0,0.3,0.0,0.5,0.0,...,2.7,1.5,0.3,4.7,1.7,0.0,0.0,0.0,0.3,0.2
1,ALLIED GARDENS,0.0,0.1,0.1,1.1,2.6,0.1,0.0,0.0,0.1,...,2.0,1.3,0.8,2.6,1.1,0.0,0.0,0.0,0.3,0.0
2,ALTA VISTA,0.0,0.0,0.0,1.1,3.3,0.4,0.0,0.0,0.0,...,3.0,0.7,0.0,1.9,0.4,0.0,0.0,0.0,0.0,0.0
3,AZALEA/HOLLYWOOD PARK,0.0,0.0,0.6,3.0,7.3,0.3,0.3,0.9,0.3,...,4.6,3.0,1.5,4.0,3.4,0.0,0.3,0.0,2.4,0.0
4,BALBOA PARK,0.0,4.3,14.9,51.2,89.6,10.7,0.0,19.2,2.1,...,93.8,2.1,6.4,74.6,89.6,0.0,0.0,0.0,2.1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,UNIVERSITY CITY,0.0,0.1,0.4,1.0,2.1,0.2,0.0,0.3,0.0,...,2.1,2.2,0.3,3.0,1.0,0.0,0.0,0.0,0.1,0.0
122,UNIVERSITY HEIGHTS,0.1,0.1,0.1,1.9,2.5,0.3,0.0,0.3,0.1,...,3.1,2.0,0.4,3.9,1.1,0.0,0.0,0.1,0.0,0.0
123,VALENCIA PARK,0.0,0.1,0.6,5.3,6.8,0.6,0.0,0.2,0.4,...,4.3,1.7,1.3,5.4,2.2,0.0,0.0,0.0,1.3,0.1
124,WOODED AREA,0.0,0.0,0.3,0.0,0.3,0.0,0.0,0.0,0.0,...,1.4,0.3,0.0,1.7,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
crime_df_2023.head()
crime_df_2023 = crime_df_2023.iloc[:-1]
crime_df_2023.head()
# upload_cleaned_crime_df(crime_df_2023, '2023')

,Neighborhood,Homicide,Kidnapping/Abduction,Sexual Assault,Aggravated Assault,Simple Assault,Intimidation,Other Sex Offenses,Robbery,Arson,...,Motor Vehicle Theft,Fraud/Counterfeiting,Stolen Property Offenses,Vandalism,Drug/Narcotic Offenses,Pornography/Obscene Material,Gambling Offenses,Prostitution,Weapon Law Violations,Animal Cruelty
0,ADAMS NORTH,0.0,0.0,0.0,1.5,3.0,0.3,0.0,0.5,0.0,...,2.7,1.5,0.3,4.7,1.7,0.0,0.0,0.0,0.3,0.2
1,ALLIED GARDENS,0.0,0.1,0.1,1.1,2.6,0.1,0.0,0.0,0.1,...,2.0,1.3,0.8,2.6,1.1,0.0,0.0,0.0,0.3,0.0
2,ALTA VISTA,0.0,0.0,0.0,1.1,3.3,0.4,0.0,0.0,0.0,...,3.0,0.7,0.0,1.9,0.4,0.0,0.0,0.0,0.0,0.0
3,AZALEA/HOLLYWOOD PARK,0.0,0.0,0.6,3.0,7.3,0.3,0.3,0.9,0.3,...,4.6,3.0,1.5,4.0,3.4,0.0,0.3,0.0,2.4,0.0
4,BALBOA PARK,0.0,4.3,14.9,51.2,89.6,10.7,0.0,19.2,2.1,...,93.8,2.1,6.4,74.6,89.6,0.0,0.0,0.0,2.1,0.0


In [15]:
upload_cleaned_crime_df(crime_df_2023, 2023)
upload_cleaned_crime_df(crime_df_2024, 2024)
upload_cleaned_crime_df(crime_df_2025, 2025)


Data successfully saved to "data/02-processed/clean_nibrs_crime_rates_2023.csv"
Data successfully saved to "data/02-processed/clean_nibrs_crime_rates_2024.csv"
Data successfully saved to "data/02-processed/clean_nibrs_crime_rates_2025.csv"
